### 1. Install Required Libraries
First, we need to install the Hugging Face `transformers` and `datasets` libraries, along with `accelerate` for efficient training.

In [1]:
!pip install -q transformers datasets accelerate

### 2. Load Model and Tokenizer
We will initialize the GPT-2 model and its tokenizer. Since GPT-2 doesn't have a default padding token, we will configure one.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT2 does not have a padding token by default, so we set it to the EOS token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### 3. Basic Text Generation (Inference)
Let's test the pre-trained model with a simple prompt before any custom fine-tuning.

In [3]:
prompt = "In a shocking turn of events, AI has learned to"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In a shocking turn of events, AI has learned to recognise human faces.

The AI will recognise the person's name, face, and social media profile. The AI can then read and interpret the information in their head. This is how the


### 4. Load and Prepare Custom Dataset
For this example, we will load a small text-based dataset (like `wikitext`), block the texts into chunks of a specific block size (e.g., 128 tokens), and tokenize them.

In [6]:
from datasets import Dataset

# Creating an explicit custom dataset locally to avoid Hub connection/URI issues
custom_data = {
    "text": [
        "Artificial intelligence is transforming the landscape of modern technology and research.",
        "Natural language processing allows computers to understand and generate human language.",
        "Fine-tuning language models on custom domain text improves their generation relevance.",
        "GPT-2 is a transformer-based model capable of generating coherent paragraphs of text.",
        "Machine learning workflows involve data preprocessing, tokenization, training, and evaluation.",
        "To train a custom language model, one needs high-quality domain-specific text resources."
    ]
}

dataset = Dataset.from_dict(custom_data)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Data collator for language modeling (automatically handles padding and creates labels)
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

### 5. Configure Trainer and Fine-tune
We will now set up the training arguments and use the `Trainer` class to run a quick fine-tuning epoch on our custom data.

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=50,
    save_total_limit=1,
    logging_steps=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=False, # Set to False for CPU safety; set to True if T4 GPU is active
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.111752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9, training_loss=3.830025249057346, metrics={'train_runtime': 60.1256, 'train_samples_per_second': 0.299, 'train_steps_per_second': 0.15, 'total_flos': 1175814144000.0, 'train_loss': 3.830025249057346, 'epoch': 3.0})

### 6. Generate Text with the Fine-tuned Model
Let's test the fine-tuned model with a prompt aligned with the custom training data (e.g., machine learning or natural language processing) to observe how its generations have adapted.

In [8]:
test_prompt = "Natural language processing is"
inputs = tokenizer(test_prompt, return_tensors="pt")

# Move model to CPU or GPU depending on environment
device = "cuda" if model.device.type == "cuda" else "cpu"
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7
)

print("Generated text:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Generated text:
Natural language processing is known as neural processing. It involves a process that involves processing information. In a neural system, information is presented to a-process. The information presented in a brain is then presented as a representation of a text. A neural network
